# Sub-agent architectures

This notebook implements sub-agent architectures where each sub-agent is an autonomous system with its own internal decision-making loop. Unlike simple LLM-call nodes, these sub-agents:
- **Have their own state** — each sub-agent maintains separate state from the coordinator and other sub-agents.
- **Run internal loops** — each sub-agent iterates (draft → reflect → decide) until satisfied with its output.
- **Make autonomous decisions** — each sub-agent decides when its work is complete, not the coordinator.
- **Are implemented as LangGraph subgraphs** — each sub-agent is a compiled `StateGraph` that the coordinator invokes.

The coordinator orchestrates these autonomous sub-agents but does not control their internal processes. When the coordinator calls a sub-agent, it hands off control entirely - the sub-agent runs its complete internal loop and returns only when it has decided its task is complete.

This is how production multi-agent systems are actually built: specialized agents with autonomy, coordinated at the task level but independent in execution.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
import os

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## Pattern 1 - Hierarchical coordination with autonomous sub-agents
We will build three autonomous sub-agents - planner, executor, and critic - each as its own complete graph with an internal refinement loop. The planner iterates on its plan until satisfied, the executor iterates on implementation, and the critic iterates on evaluation. A coordinator graph orchestrates these sub-agents but does not interfere with their internal processes.

Each sub-agent follows the pattern: **draft → reflect → route**. The routing function decides whether to loop back for another iteration or finalize and return. This is genuine autonomy - the sub-agent controls its own execution until it meets its quality criteria.

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# SUB-AGENT 1: PlannerAgent (autonomous planning agent with internal loop)
# ═══════════════════════════════════════════════════════════════════════════

class PlannerState(TypedDict):
    """State owned by the PlannerAgent sub-agent — isolated from coordinator state."""
    task: str           # Input: the task to plan for
    draft_plan: str     # Current plan draft (updated each iteration)
    reflection: str     # Self-critique of current draft
    iterations: int     # Loop counter
    final_plan: str     # Output: the polished final plan

PLANNER_PROMPT = (
    "You are a planning specialist. Develop clear, actionable plans "
    "with concrete steps, dependencies, and risk considerations."
)

def draft_plan_node(state: PlannerState) -> dict:
    """
    Creates or refines the plan based on prior reflection.
    This node is called multiple times in the sub-agent's internal loop.
    """
    system = SystemMessage(content=PLANNER_PROMPT)

    if state.get("reflection"):
        # Refinement pass: address issues from reflection
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Current plan:\n{state['draft_plan']}\n\n"
            f"Reflection on issues:\n{state['reflection']}\n\n"
            f"Refine the plan to address these issues."
        )
    else:
        # Initial pass: create first draft
        prompt = f"Task: {state['task']}\n\nDevelop a detailed plan."

    response = llm.invoke([system, HumanMessage(content=prompt)])
    return {
        "draft_plan": response.content,
        "iterations": state.get("iterations", 0) + 1
    }

def reflect_on_plan_node(state: PlannerState) -> dict:
    """
    Self-critique: identifies gaps, risks, or unclear steps in the current draft.
    This reflection drives the decision to refine or finalize.
    """
    prompt = (
        f"Evaluate this plan critically:\n{state['draft_plan']}\n\n"
        f"What's missing? What's unclear? What risks aren't addressed?\n"
        f"Reply with specific issues or 'COMPLETE' if the plan is thorough."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"reflection": response.content}

def route_planner(state: PlannerState) -> Literal["refine", "finalize"]:
    """
    Routing function: the sub-agent's autonomous decision point.
    Decides whether to iterate again (refine) or return the result (finalize).
    This is what makes the sub-agent autonomous — it decides when it's done.
    """
    # Hard cap to prevent infinite loops
    if state.get("iterations", 0) >= 2:
        return "finalize"

    # If reflection indicates completion, finalize
    if "COMPLETE" in state.get("reflection", "").upper():
        return "finalize"

    # Otherwise, refine
    return "refine"

def finalize_plan_node(state: PlannerState) -> dict:
    """Marks the plan as final and ready to return to the coordinator."""
    return {"final_plan": state["draft_plan"]}

print("PlannerAgent nodes defined: draft → reflect → route → finalize")

PlannerAgent nodes defined: draft → reflect → route → finalize


In [4]:
# Build the PlannerAgent as a complete subgraph
planner_builder = StateGraph(PlannerState)

planner_builder.add_node("draft",    draft_plan_node)
planner_builder.add_node("reflect",  reflect_on_plan_node)
planner_builder.add_node("finalize", finalize_plan_node)

# Flow: START → draft → reflect → route decision
planner_builder.add_edge(START, "draft")
planner_builder.add_edge("draft", "reflect")

# Conditional edge creates the internal loop
planner_builder.add_conditional_edges(
    "reflect",
    route_planner,
    {
        "refine":   "draft",     # Loop back: draft again with reflection context
        "finalize": "finalize"   # Exit loop: finalize and return
    }
)

planner_builder.add_edge("finalize", END)

# Compile into an autonomous agent — this is a complete runnable sub-agent
planner_agent = planner_builder.compile()

print("✓ PlannerAgent compiled as autonomous subgraph")
print("  Internal loop: draft → reflect → [refine or finalize]")

✓ PlannerAgent compiled as autonomous subgraph
  Internal loop: draft → reflect → [refine or finalize]


In [5]:
# Test the PlannerAgent in isolation — it runs its complete internal loop autonomously
print("Testing PlannerAgent autonomously:\n")
print("=" * 60)

planner_result = planner_agent.invoke({
    "task": "Design a user authentication system with email/password login and session management",
    "draft_plan": "",
    "reflection": "",
    "iterations": 0,
    "final_plan": ""
})

print(f"Planner ran {planner_result['iterations']} iteration(s)")
print(f"\nFinal plan:\n{planner_result['final_plan'][:400]}...")

Testing PlannerAgent autonomously:

Planner ran 2 iteration(s)

Final plan:
### Refined User Authentication System Plan

#### Objective:
To design a secure and user-friendly user authentication system that allows users to log in using their email and password, while effectively managing user sessions and enhancing security.

---

### Step 1: Requirements Gathering

**Action Items:**
1. Identify user roles (e.g., Admin, Regular User).
2. Define security requirements (e.g.,...


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# SUB-AGENT 2: ExecutorAgent (autonomous implementation agent)
# ═══════════════════════════════════════════════════════════════════════════

class ExecutorState(TypedDict):
    plan: str               # Input from coordinator
    draft_execution: str    # Current implementation draft
    verification: str       # Self-check of implementation quality
    iterations: int
    final_execution: str    # Output to coordinator

EXECUTOR_PROMPT = (
    "You are an implementation specialist. Produce detailed, concrete output "
    "that fully implements the given plan."
)

def draft_execution_node(state: ExecutorState) -> dict:
    system = SystemMessage(content=EXECUTOR_PROMPT)

    if state.get("verification"):
        prompt = (
            f"Plan: {state['plan']}\n\n"
            f"Current implementation:\n{state['draft_execution']}\n\n"
            f"Issues found:\n{state['verification']}\n\n"
            f"Revise the implementation to fix these issues."
        )
    else:
        prompt = f"Plan: {state['plan']}\n\nImplement this plan in detail."

    response = llm.invoke([system, HumanMessage(content=prompt)])
    return {
        "draft_execution": response.content,
        "iterations": state.get("iterations", 0) + 1
    }

def verify_execution_node(state: ExecutorState) -> dict:
    prompt = (
        f"Verify this implementation against the plan:\n{state['draft_execution']}\n\n"
        f"Original plan: {state['plan']}\n\n"
        f"Are all steps implemented? Any gaps or errors? Reply with issues or 'COMPLETE'."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"verification": response.content}

def route_executor(state: ExecutorState) -> Literal["refine", "finalize"]:
    if state.get("iterations", 0) >= 2:
        return "finalize"
    if "COMPLETE" in state.get("verification", "").upper():
        return "finalize"
    return "refine"

def finalize_execution_node(state: ExecutorState) -> dict:
    return {"final_execution": state["draft_execution"]}

# Build ExecutorAgent subgraph
executor_builder = StateGraph(ExecutorState)
executor_builder.add_node("draft",    draft_execution_node)
executor_builder.add_node("verify",   verify_execution_node)
executor_builder.add_node("finalize", finalize_execution_node)

executor_builder.add_edge(START, "draft")
executor_builder.add_edge("draft", "verify")
executor_builder.add_conditional_edges("verify", route_executor, {
    "refine": "draft",
    "finalize": "finalize"
})
executor_builder.add_edge("finalize", END)

executor_agent = executor_builder.compile()
print("✓ ExecutorAgent compiled as autonomous subgraph")

✓ ExecutorAgent compiled as autonomous subgraph


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# SUB-AGENT 3: CriticAgent (autonomous evaluation agent)
# ═══════════════════════════════════════════════════════════════════════════

class CriticState(TypedDict):
    task: str               # Original task for context
    execution: str          # Implementation to evaluate
    draft_critique: str     # Current critique draft
    reflection: str         # Self-check: is critique comprehensive?
    iterations: int
    final_critique: str     # Output to coordinator

CRITIC_PROMPT = (
    "You are an evaluation specialist. Assess work critically: identify gaps, "
    "risks, edge cases, and improvement opportunities. Be specific and constructive."
)

def draft_critique_node(state: CriticState) -> dict:
    system = SystemMessage(content=CRITIC_PROMPT)

    if state.get("reflection"):
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Implementation:\n{state['execution']}\n\n"
            f"Current critique:\n{state['draft_critique']}\n\n"
            f"Gaps in critique:\n{state['reflection']}\n\n"
            f"Expand the critique to address these gaps."
        )
    else:
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Implementation:\n{state['execution']}\n\n"
            f"Provide a thorough critique."
        )

    response = llm.invoke([system, HumanMessage(content=prompt)])
    return {
        "draft_critique": response.content,
        "iterations": state.get("iterations", 0) + 1
    }

def reflect_on_critique_node(state: CriticState) -> dict:
    prompt = (
        f"Is this critique comprehensive?\n{state['draft_critique']}\n\n"
        f"What aspects are not covered? Reply with gaps or 'COMPLETE'."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"reflection": response.content}

def route_critic(state: CriticState) -> Literal["refine", "finalize"]:
    if state.get("iterations", 0) >= 2:
        return "finalize"
    if "COMPLETE" in state.get("reflection", "").upper():
        return "finalize"
    return "refine"

def finalize_critique_node(state: CriticState) -> dict:
    return {"final_critique": state["draft_critique"]}

# Build CriticAgent subgraph
critic_builder = StateGraph(CriticState)
critic_builder.add_node("draft",    draft_critique_node)
critic_builder.add_node("reflect",  reflect_on_critique_node)
critic_builder.add_node("finalize", finalize_critique_node)

critic_builder.add_edge(START, "draft")
critic_builder.add_edge("draft", "reflect")
critic_builder.add_conditional_edges("reflect", route_critic, {
    "refine": "draft",
    "finalize": "finalize"
})
critic_builder.add_edge("finalize", END)

critic_agent = critic_builder.compile()
print("✓ CriticAgent compiled as autonomous subgraph")

✓ CriticAgent compiled as autonomous subgraph


With three autonomous sub-agents compiled, we now build the **coordinator graph**. The coordinator's nodes invoke the sub-agents as complete units - calling `planner_agent.invoke()` hands control to the planner, which runs its entire internal loop and returns only when it has decided the plan is complete. The coordinator orchestrates these autonomous agents but does not control or observe their internal iterations.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# COORDINATOR GRAPH: Orchestrates autonomous sub-agents
# ═══════════════════════════════════════════════════════════════════════════

class HierarchicalState(TypedDict):
    """Coordinator's state — high-level task coordination only."""
    task: str
    plan: str         # Output from PlannerAgent
    execution: str    # Output from ExecutorAgent
    critique: str     # Output from CriticAgent
    final: str        # Integrated final answer

def coordinator_planner_node(state: HierarchicalState) -> dict:
    """
    Invokes the PlannerAgent sub-agent.
    The sub-agent runs its complete internal loop autonomously.
    """
    print("  [Coordinator] Invoking PlannerAgent...")

    # Hand off to the autonomous planner sub-agent
    planner_result = planner_agent.invoke({
        "task": state["task"],
        "draft_plan": "",
        "reflection": "",
        "iterations": 0,
        "final_plan": ""
    })

    print(f"  [Coordinator] PlannerAgent completed ({planner_result['iterations']} iterations)")
    return {"plan": planner_result["final_plan"]}

def coordinator_executor_node(state: HierarchicalState) -> dict:
    """Invokes the ExecutorAgent sub-agent."""
    print("  [Coordinator] Invoking ExecutorAgent...")

    executor_result = executor_agent.invoke({
        "plan": state["plan"],
        "draft_execution": "",
        "verification": "",
        "iterations": 0,
        "final_execution": ""
    })

    print(f"  [Coordinator] ExecutorAgent completed ({executor_result['iterations']} iterations)")
    return {"execution": executor_result["final_execution"]}

def coordinator_critic_node(state: HierarchicalState) -> dict:
    """Invokes the CriticAgent sub-agent."""
    print("  [Coordinator] Invoking CriticAgent...")

    critic_result = critic_agent.invoke({
        "task": state["task"],
        "execution": state["execution"],
        "draft_critique": "",
        "reflection": "",
        "iterations": 0,
        "final_critique": ""
    })

    print(f"  [Coordinator] CriticAgent completed ({critic_result['iterations']} iterations)")
    return {"critique": critic_result["final_critique"]}

def coordinator_integration_node(state: HierarchicalState) -> dict:
    """Synthesizes all sub-agent outputs into a final answer."""
    print("  [Coordinator] Integrating sub-agent outputs...")

    prompt = (
        f"Original task: {state['task']}\n\n"
        f"Plan:\n{state['plan']}\n\n"
        f"Implementation:\n{state['execution']}\n\n"
        f"Critique:\n{state['critique']}\n\n"
        f"Synthesize these into one comprehensive final answer."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final": response.content}

print("Coordinator nodes defined (each invokes an autonomous sub-agent)")

Coordinator nodes defined (each invokes an autonomous sub-agent)


In [9]:
# Build the coordinator graph (orchestrates sub-agents)
coordinator_builder = StateGraph(HierarchicalState)

coordinator_builder.add_node("planner",  coordinator_planner_node)
coordinator_builder.add_node("executor", coordinator_executor_node)
coordinator_builder.add_node("critic",   coordinator_critic_node)
coordinator_builder.add_node("integrate", coordinator_integration_node)

# Sequential orchestration: each sub-agent completes before the next is invoked
coordinator_builder.add_edge(START,       "planner")
coordinator_builder.add_edge("planner",   "executor")
coordinator_builder.add_edge("executor",  "critic")
coordinator_builder.add_edge("critic",    "integrate")
coordinator_builder.add_edge("integrate", END)

hierarchical_system = coordinator_builder.compile()

print("✓ Coordinator graph compiled")
print("  Flow: planner sub-agent → executor sub-agent → critic sub-agent → integrate")

✓ Coordinator graph compiled
  Flow: planner sub-agent → executor sub-agent → critic sub-agent → integrate


In [10]:
task = (
    "Design a user authentication system for a web application. "
    "Support email/password login, password reset, and session management."
)

print(f"Task: {task}\n")
print("=" * 60)
print("Running hierarchical system with autonomous sub-agents:\n")

result = hierarchical_system.invoke({
    "task": task,
    "plan": "",
    "execution": "",
    "critique": "",
    "final": ""
})

print("\n" + "=" * 60)
print("Final integrated result:")
print(result["final"])

Task: Design a user authentication system for a web application. Support email/password login, password reset, and session management.

Running hierarchical system with autonomous sub-agents:

  [Coordinator] Invoking PlannerAgent...
  [Coordinator] PlannerAgent completed (2 iterations)
  [Coordinator] Invoking ExecutorAgent...
  [Coordinator] ExecutorAgent completed (2 iterations)
  [Coordinator] Invoking CriticAgent...
  [Coordinator] CriticAgent completed (2 iterations)
  [Coordinator] Integrating sub-agent outputs...

Final integrated result:
### Comprehensive User Authentication System Design and Implementation

#### Objective:
To design a secure, efficient, and user-friendly user authentication system for a web application that supports email/password login, password reset, session management, and enhanced security features, while ensuring compliance with relevant regulations.

---

### Step 1: Requirements Gathering

**Action Items:**
1. Identify user roles (e.g., Admin, User).


**What makes these sub-agents autonomous**

**Separate state ownership.** Each sub-agent (`PlannerAgent`, `ExecutorAgent`, `CriticAgent`) has its own `TypedDict` state that the coordinator never sees. When the coordinator invokes a sub-agent, it provides input fields and receives output fields — the sub-agent's internal state (drafts, reflections, iteration counters) remains private.

**Internal decision-making loop.** Each sub-agent contains a conditional edge that creates a cycle: `draft → reflect → route → [refine or finalize]`. The routing function is the sub-agent's autonomous decision point — it evaluates whether its output meets quality criteria and decides whether to iterate again or return. The coordinator does not participate in this decision; the sub-agent is in full control.

**Compiled as independent executables.** Each sub-agent is compiled via `.compile()` into a complete LangGraph runnable. The coordinator does not call individual nodes within the sub-agent; it invokes the entire compiled sub-agent as a black box via `sub_agent.invoke()`. This is genuine encapsulation — the sub-agent's internal structure is opaque to the coordinator.

**Contrast with the previous implementation.** In the earlier notebook, "agents" were single-function nodes that executed once and returned immediately. The "agent loop" was the graph's execution loop, not any individual agent's internal loop. Here, each sub-agent has its own loop and decides autonomously when it is satisfied with its output. This is the distinction between **orchestrated nodes** and **autonomous sub-agents**.

## Pattern 2 — Peer collaboration with autonomous sub-agents

In peer collaboration, each peer is an autonomous agent that internally refines its contribution before posting to the shared space. A peer agent reads prior contributions, drafts its response, reflects on whether it adds genuine value, and iterates until satisfied — then posts the polished result. The coordinator invokes peers sequentially but each peer's internal refinement is autonomous.

In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# SUB-AGENT: PeerAgent (autonomous collaborative agent)
# ═══════════════════════════════════════════════════════════════════════════

class PeerAgentState(TypedDict):
    topic: str                  # Shared topic
    prior_contributions: str    # What other peers have said
    draft_contribution: str     # This peer's current draft
    reflection: str             # Self-assessment of contribution value
    iterations: int
    final_contribution: str     # Polished output to post

def create_peer_agent(name: str, expertise: str):
    """
    Factory that creates an autonomous peer agent subgraph.
    Each peer has its own internal refinement loop before posting.
    """

    def draft_contribution_node(state: PeerAgentState) -> dict:
        system = SystemMessage(content=f"You are a {expertise} expert.")

        if state.get("reflection"):
            prompt = (
                f"Topic: {state['topic']}\n\n"
                f"Prior discussion:\n{state['prior_contributions']}\n\n"
                f"Your current draft:\n{state['draft_contribution']}\n\n"
                f"Feedback: {state['reflection']}\n\n"
                f"Revise your contribution to add more value."
            )
        else:
            prompt = (
                f"Topic: {state['topic']}\n\n"
                f"Prior discussion:\n{state['prior_contributions'] or 'None yet'}\n\n"
                f"Add ONE insightful contribution from your {expertise} perspective. "
                f"Build on what's been said."
            )

        response = llm.invoke([system, HumanMessage(content=prompt)])
        return {
            "draft_contribution": response.content,
            "iterations": state.get("iterations", 0) + 1
        }

    def reflect_on_contribution_node(state: PeerAgentState) -> dict:
        prompt = (
            f"Does this contribution add genuine value?\n{state['draft_contribution']}\n\n"
            f"Given prior discussion: {state['prior_contributions']}\n\n"
            f"Reply with issues or 'VALUABLE' if it adds real insight."
        )
        response = llm.invoke([HumanMessage(content=prompt)])
        return {"reflection": response.content}

    def route_peer(state: PeerAgentState) -> Literal["refine", "finalize"]:
        if state.get("iterations", 0) >= 2:
            return "finalize"
        if "VALUABLE" in state.get("reflection", "").upper():
            return "finalize"
        return "refine"

    def finalize_contribution_node(state: PeerAgentState) -> dict:
        return {"final_contribution": f"[{name}]: {state['draft_contribution']}"}

    # Build the peer agent subgraph
    builder = StateGraph(PeerAgentState)
    builder.add_node("draft", draft_contribution_node)
    builder.add_node("reflect", reflect_on_contribution_node)
    builder.add_node("finalize", finalize_contribution_node)

    builder.add_edge(START, "draft")
    builder.add_edge("draft", "reflect")
    builder.add_conditional_edges("reflect", route_peer, {
        "refine": "draft",
        "finalize": "finalize"
    })
    builder.add_edge("finalize", END)

    return builder.compile()

# Create three autonomous peer agents
security_agent = create_peer_agent("SecurityEngineer", "application security")
ux_agent       = create_peer_agent("UXDesigner", "user experience")
infra_agent    = create_peer_agent("InfraEngineer", "cloud infrastructure")

print("✓ Three autonomous peer agents created")

✓ Three autonomous peer agents created


In [12]:
class PeerCollaborationState(TypedDict):
    topic: str
    contributions: list[str]  # Accumulates contributions from all peers

def invoke_peer_node(peer_agent, name: str):
    """Creates a coordinator node that invokes one autonomous peer agent."""
    def node(state: PeerCollaborationState) -> dict:
        print(f"  [Coordinator] Invoking {name}...")

        # Format prior contributions for this peer
        prior = "\n".join(state.get("contributions", []))

        # Invoke the peer sub-agent — it runs its internal refinement loop
        peer_result = peer_agent.invoke({
            "topic": state["topic"],
            "prior_contributions": prior,
            "draft_contribution": "",
            "reflection": "",
            "iterations": 0,
            "final_contribution": ""
        })

        print(f"  [Coordinator] {name} completed ({peer_result['iterations']} iterations)")

        # Add the peer's polished contribution
        existing = state.get("contributions", [])
        return {"contributions": existing + [peer_result["final_contribution"]]}

    return node

# Build peer collaboration coordinator
peer_coord_builder = StateGraph(PeerCollaborationState)
peer_coord_builder.add_node("security", invoke_peer_node(security_agent, "SecurityEngineer"))
peer_coord_builder.add_node("ux",       invoke_peer_node(ux_agent, "UXDesigner"))
peer_coord_builder.add_node("infra",    invoke_peer_node(infra_agent, "InfraEngineer"))

peer_coord_builder.add_edge(START, "security")
peer_coord_builder.add_edge("security", "ux")
peer_coord_builder.add_edge("ux", "infra")
peer_coord_builder.add_edge("infra", END)

peer_system = peer_coord_builder.compile()
print("✓ Peer collaboration coordinator compiled")

✓ Peer collaboration coordinator compiled


In [13]:
topic = "Design a two-factor authentication (2FA) system for a mobile banking app."

print(f"Topic: {topic}\n")
print("=" * 60)
print("Running peer collaboration with autonomous agents:\n")

peer_result = peer_system.invoke({
    "topic": topic,
    "contributions": []
})

print("\n" + "=" * 60)
print(f"Collected {len(peer_result['contributions'])} contributions:")
for contribution in peer_result["contributions"]:
    print(f"\n{contribution}")

Topic: Design a two-factor authentication (2FA) system for a mobile banking app.

Running peer collaboration with autonomous agents:

  [Coordinator] Invoking SecurityEngineer...
  [Coordinator] SecurityEngineer completed (1 iterations)
  [Coordinator] Invoking UXDesigner...
  [Coordinator] UXDesigner completed (1 iterations)
  [Coordinator] Invoking InfraEngineer...
  [Coordinator] InfraEngineer completed (1 iterations)

Collected 3 contributions:

[SecurityEngineer]: When designing a two-factor authentication (2FA) system for a mobile banking app, one critical aspect to consider is the implementation of adaptive authentication mechanisms. 

**Insightful Contribution: Adaptive Authentication**

Adaptive authentication enhances the security of the 2FA process by assessing the risk level of each login attempt based on various contextual factors. For instance, the system can evaluate parameters such as the user's location, device, time of access, and behavioral patterns. If a login attem

## Pattern 3 — Pipeline processing with autonomous sub-agents
In the pipeline, each stage is an autonomous agent that internally refines its output before passing it downstream. A research agent iterates on gathering comprehensive information, an analysis agent iterates on identifying patterns, and so on. Each stage internally loops until satisfied with its quality, then outputs to the next stage.

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# PIPELINE SUB-AGENT: Generic stage agent with internal refinement loop
# ═══════════════════════════════════════════════════════════════════════════

class PipelineStageState(TypedDict):
    input_context: str      # From previous stage
    draft_output: str       # Current output draft
    reflection: str         # Quality assessment
    iterations: int
    final_output: str       # Polished output for next stage

def create_pipeline_stage_agent(stage_name: str, stage_instruction: str):
    """Factory for autonomous pipeline stage agents."""

    def draft_output_node(state: PipelineStageState) -> dict:
        system = SystemMessage(content=f"You are the {stage_name} stage in a pipeline.")

        if state.get("reflection"):
            prompt = (
                f"Input: {state['input_context']}\n\n"
                f"Current output:\n{state['draft_output']}\n\n"
                f"Issues: {state['reflection']}\n\n"
                f"Revise to address these issues."
            )
        else:
            prompt = f"Input: {state['input_context']}\n\n{stage_instruction}"

        response = llm.invoke([system, HumanMessage(content=prompt)])
        return {
            "draft_output": response.content,
            "iterations": state.get("iterations", 0) + 1
        }

    def reflect_on_output_node(state: PipelineStageState) -> dict:
        prompt = (
            f"Is this {stage_name} output thorough and complete?\n{state['draft_output']}\n\n"
            f"Reply with gaps or 'COMPLETE'."
        )
        response = llm.invoke([HumanMessage(content=prompt)])
        return {"reflection": response.content}

    def route_stage(state: PipelineStageState) -> Literal["refine", "finalize"]:
        if state.get("iterations", 0) >= 2:
            return "finalize"
        if "COMPLETE" in state.get("reflection", "").upper():
            return "finalize"
        return "refine"

    def finalize_output_node(state: PipelineStageState) -> dict:
        return {"final_output": state["draft_output"]}

    builder = StateGraph(PipelineStageState)
    builder.add_node("draft", draft_output_node)
    builder.add_node("reflect", reflect_on_output_node)
    builder.add_node("finalize", finalize_output_node)

    builder.add_edge(START, "draft")
    builder.add_edge("draft", "reflect")
    builder.add_conditional_edges("reflect", route_stage, {"refine": "draft", "finalize": "finalize"})
    builder.add_edge("finalize", END)

    return builder.compile()

# Create four autonomous pipeline stage agents
research_agent = create_pipeline_stage_agent(
    "Research",
    "Gather and organize comprehensive facts and context about this topic."
)
analysis_agent = create_pipeline_stage_agent(
    "Analysis",
    "Identify patterns, tradeoffs, and key decision factors from the research."
)
recommendation_agent = create_pipeline_stage_agent(
    "Recommendation",
    "Produce a concrete, actionable recommendation based on the analysis."
)
review_agent = create_pipeline_stage_agent(
    "Review",
    "Critically evaluate the recommendation for risks, gaps, and assumptions."
)

print("✓ Four autonomous pipeline stage agents created")

✓ Four autonomous pipeline stage agents created


In [15]:
class PipelineCoordinatorState(TypedDict):
    input: str
    research: str
    analysis: str
    recommendation: str
    review: str

def invoke_stage_node(stage_agent, stage_name: str, input_field: str, output_field: str):
    """Creates a coordinator node that invokes one autonomous pipeline stage."""
    def node(state: PipelineCoordinatorState) -> dict:
        print(f"  [Pipeline] Running {stage_name} stage...")

        stage_result = stage_agent.invoke({
            "input_context": state[input_field],
            "draft_output": "",
            "reflection": "",
            "iterations": 0,
            "final_output": ""
        })

        print(f"  [Pipeline] {stage_name} completed ({stage_result['iterations']} iterations)")
        return {output_field: stage_result["final_output"]}

    return node

# Build pipeline coordinator
pipeline_coord_builder = StateGraph(PipelineCoordinatorState)
pipeline_coord_builder.add_node("research",       invoke_stage_node(research_agent, "Research", "input", "research"))
pipeline_coord_builder.add_node("analysis",       invoke_stage_node(analysis_agent, "Analysis", "research", "analysis"))
pipeline_coord_builder.add_node("recommendation", invoke_stage_node(recommendation_agent, "Recommendation", "analysis", "recommendation"))
pipeline_coord_builder.add_node("review",         invoke_stage_node(review_agent, "Review", "recommendation", "review"))

pipeline_coord_builder.add_edge(START, "research")
pipeline_coord_builder.add_edge("research", "analysis")
pipeline_coord_builder.add_edge("analysis", "recommendation")
pipeline_coord_builder.add_edge("recommendation", "review")
pipeline_coord_builder.add_edge("review", END)

pipeline_system = pipeline_coord_builder.compile()
print("✓ Pipeline coordinator compiled")

✓ Pipeline coordinator compiled


In [16]:
question = (
    "Should our company adopt serverless architecture "
    "for our new customer data processing platform?"
)

print(f"Input: {question}\n")
print("=" * 60)
print("Running pipeline with autonomous stage agents:\n")

pipeline_result = pipeline_system.invoke({
    "input": question,
    "research": "",
    "analysis": "",
    "recommendation": "",
    "review": ""
})

print("\n" + "=" * 60)
print("Final review:")
print(pipeline_result["review"])

Input: Should our company adopt serverless architecture for our new customer data processing platform?

Running pipeline with autonomous stage agents:

  [Pipeline] Running Research stage...
  [Pipeline] Research completed (1 iterations)
  [Pipeline] Running Analysis stage...
  [Pipeline] Analysis completed (1 iterations)
  [Pipeline] Running Recommendation stage...
  [Pipeline] Recommendation completed (1 iterations)
  [Pipeline] Running Review stage...
  [Pipeline] Review completed (1 iterations)

Final review:
### Critical Evaluation of the Recommendation

The recommendation to adopt a serverless architecture for the customer data processing platform presents several advantages, but it also carries inherent risks, gaps, and assumptions that need to be critically evaluated.

#### Risks

1. **Vendor Lock-In**:
   - The recommendation emphasizes evaluating vendor options to mitigate vendor lock-in, but the risk remains significant. Once a serverless architecture is implemented with a s

**What we implemented:**
- **Autonomous sub-agents** - Each sub-agent (`PlannerAgent`, `ExecutorAgent`, `CriticAgent`, peer agents, pipeline stage agents) is a compiled LangGraph with its own state and internal loop.
- **Internal decision-making** - Each sub-agent has a routing function that decides when to iterate (refine) or return (finalize) based on quality assessment.
- **State isolation** - Sub-agent internal state (drafts, reflections, iterations) is never visible to the coordinator.
- **True autonomy** - The coordinator invokes sub-agents as black boxes; sub-agents control their own execution and decide when they are done.

**When to use each approach:** When sub-tasks require iterative refinement, quality criteria are complex, or sub-agents need independence. More overhead but higher output quality and true autonomy.

This implementation demonstrates production-grade multi-agent systems where sub-agents are not just specialized functions, but autonomous systems with their own decision-making capabilities.